# Breast Cancer Diagnosis with Machine Learning

**Author:** Bilal Ahmad — Healthcare Data Analyst

> **Note:** This notebook is for learning and portfolio demonstration only. It is **not** clinical or medical advice, and it uses **no real patient data**. Dataset: the built-in scikit-learn breast cancer dataset (diagnostic measurements from digitised biopsy images, provided for teaching purposes).


## 1. Introduction

This notebook builds two simple classifiers — Logistic Regression and Random Forest — that predict whether a breast mass is malignant or benign from numeric diagnostic measurements. The goal is to practise a complete ML workflow: data loading, exploratory analysis, train/test splitting, model training, evaluation, and interpretation. Nothing here is a diagnostic tool.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
print("Dataset shape:", X.shape)
print("Target labels:", dict(zip(data.target_names, [0, 1])))
print("\nTarget distribution (0 = malignant, 1 = benign):")
print(y.value_counts())
print("\nMissing values in features:", int(X.isna().sum().sum()))

Dataset shape: (569, 30)
Target labels: {np.str_('malignant'): 0, np.str_('benign'): 1}

Target distribution (0 = malignant, 1 = benign):
target
1    357
0    212
Name: count, dtype: int64

Missing values in features: 0


## 2. Exploratory Data Analysis

Summary statistics show the scale of each measurement. Note how widely the features differ in scale — that is why the linear model below is trained on standardised features.

In [2]:
X.describe().T[["mean", "std", "min", "max"]].round(2)

,mean,std,min,max
mean radius,14.13,3.52,6.98,28.11
mean texture,19.29,4.30,9.71,39.28
mean perimeter,91.97,24.30,43.79,188.50
mean area,654.89,351.91,143.50,2501.00
mean smoothness,0.10,0.01,0.05,0.16
mean compactness,0.10,0.05,0.02,0.35
mean concavity,0.09,0.08,0.00,0.43
mean concave points,0.05,0.04,0.00,0.20
mean symmetry,0.18,0.03,0.11,0.30
mean fractal dimension,0.06,0.01,0.05,0.10


## 3. Train / Test Split

A stratified 80/20 split keeps the malignant/benign ratio the same in both sets, so the test score is a fair estimate.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

Train size: (455, 30) | Test size: (114, 30)


## 4. Model Training

**Logistic Regression** is a simple, interpretable linear baseline. **Random Forest** is an ensemble of decision trees that can capture non-linear patterns and also reports which features mattered most.

In [4]:
lr = LogisticRegression(max_iter=5000, random_state=42)
lr.fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
print("Both models trained.")

Both models trained.


## 5. Evaluation

Accuracy alone is not enough in a medical context: **recall** matters because a missed malignant case (false negative) is costly, and **precision** matters because false alarms cause unnecessary follow-up. The confusion matrix shows all four outcomes.

In [5]:
for name, model, Xt in [("Logistic Regression", lr, X_test_s), ("Random Forest", rf, X_test)]:
    pred = model.predict(Xt)
    print(f"--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_test, pred), 4))
    print("Precision:", round(precision_score(y_test, pred), 4))
    print("Recall   :", round(recall_score(y_test, pred), 4))
    print("Confusion matrix (rows = actual, cols = predicted):")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=data.target_names))

--- Logistic Regression ---
Accuracy : 0.9825
Precision: 0.9861
Recall   : 0.9861
Confusion matrix (rows = actual, cols = predicted):
[[41  1]
 [ 1 71]]
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

--- Random Forest ---
Accuracy : 0.9474
Precision: 0.9583
Recall   : 0.9583
Confusion matrix (rows = actual, cols = predicted):
[[39  3]
 [ 3 69]]
              precision    recall  f1-score   support

   malignant       0.93      0.93      0.93        42
      benign       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



## 6. Feature Importance (Random Forest)

The chart below shows which measurements the Random Forest relied on most. This is useful for discussion, not proof that a feature *causes* cancer.

In [6]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top10 = importances.head(10)
print(top10.round(4))
top10.sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 most important features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

worst perimeter         0.1373
worst area              0.1373
worst concave points    0.1155
mean concave points     0.0918
worst radius            0.0841
mean radius             0.0627
mean perimeter          0.0497
mean concavity          0.0441
mean area               0.0405
worst concavity         0.0348
dtype: float64


## 7. Conclusion

**Results (from running the code above):** on the 114-sample test set (42 malignant, 72 benign), Logistic Regression reached accuracy 0.9825 with precision and recall both 0.9861 (confusion matrix: 41 true malignant, 1 false benign, 1 false malignant, 71 true benign), while Random Forest reached accuracy 0.9474 with precision and recall both 0.9583. The most important Random Forest features were worst perimeter (0.1373) and worst area (0.1373), followed by worst concave points (0.1155).

Both models perform strongly on this teaching dataset, but that does not make them clinical tools: the data are curated, the classes are balanced, and real diagnosis involves clinical context far beyond these measurements. The value of this notebook is the workflow — clean evaluation, honest metrics, and checking *which* features drive predictions before trusting any model.